# Practice 7 — Persistent homology

In this practical we use **GUDHI** to explore the persistence material from Sessions 6 and 7.

The code runs as provided. Cells marked **`# TODO`** ask you to change a parameter or run a comparison; write your interpretation below each exercise.

## 0. Setup
Run once in a Python 3.11+ notebook environment, such as Colab. `POT` supplies the optimal-transport backend for Wasserstein distances.

In [ ]:
%pip install -q "gudhi==3.13.0" "POT==0.9.7.post1" numpy scipy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gudhi as gd
from gudhi.wasserstein import wasserstein_distance
from scipy.spatial.distance import pdist
from itertools import combinations
from importlib.metadata import version

SEED = 7
print("GUDHI:", version("gudhi"), "| POT:", version("POT"))
plt.rcParams.update({"figure.figsize": (7, 3.5), "font.size": 11})

## 1. A filtration by hand
We reproduce the example from Session 7:

| Time | New simplices |
|---|---|
| 0 | vertices 1, 2, 3, 4 |
| 1 | edge (1,2) |
| 2 | edges (1,3) and (2,3) |
| 3 | edge (3,4) |
| 4 | filled triangle (1,2,3) |

A filtration must satisfy $f(\tau)\leq f(\sigma)$ whenever $\tau$ is a face of $\sigma$.
`SimplexTree.insert` inserts missing faces too. Their default insertion time would be that of the simplex, so **insert the vertices and edges at their intended times first**. Inserting a coface later does not increase existing face times.

**Predict before running:** when does the loop appear, and when does it disappear?

In [ ]:
def make_filtration(fill_time=4.0):
    if fill_time < 2:
        raise ValueError("The triangle cannot precede its boundary edges.")
    st = gd.SimplexTree()
    for v in [1, 2, 3, 4]:
        st.insert([v], filtration=0.0)
    for edge, time in [([1, 2], 1), ([1, 3], 2),
                       ([2, 3], 2), ([3, 4], 3)]:
        st.insert(edge, filtration=time)
    st.insert([1, 2, 3], filtration=fill_time)
    return st

st = make_filtration()
for simplex, time in st.get_filtration():
    print(f"{str(simplex):12s} enters at t={time:g}")

In [ ]:
positions = {1: (0, 0), 2: (1, 0), 3: (.5, .9), 4: (1.5, .9)}
fig, axes = plt.subplots(1, 5, figsize=(13, 2.6))
for t, ax in enumerate(axes):
    for simplex, time in st.get_filtration():
        if time > t:
            continue
        xy = np.array([positions[v] for v in simplex])
        if len(simplex) == 3:
            ax.fill(*xy.T, color="teal", alpha=.2, zorder=0)
        elif len(simplex) == 2:
            ax.plot(*xy.T, color=".35", zorder=1)
    for v, xy in positions.items():
        ax.scatter(*xy, s=260, color="black", zorder=2)
        ax.text(*xy, str(v), color="white", ha="center", va="center", zorder=3)
    ax.set(title=f"$K_{t}$: t = {t}", xlim=(-.2, 1.7), ylim=(-.2, 1.1), aspect="equal")
    ax.axis("off")
plt.tight_layout(); plt.show()

### Barcodes and persistence diagrams
`persistence()` returns pairs `(dimension, (birth, death))`. The interval convention is **$[b,d)$**: a class is absent at its death time. Zero-length intervals are omitted by default.

An infinite interval survives in the final complex, which we hold constant after the last step. `persistence_intervals_in_dimension(k)` returns an array with one row `(birth, death)` per interval. Compute persistence before calling it.

In [ ]:
persistence = st.persistence(homology_coeff_field=2)
for k in [0, 1]:
    print(f"H{k} intervals:\n", st.persistence_intervals_in_dimension(k))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
gd.plot_persistence_barcode(persistence, axes=axes[0], legend=True)
gd.plot_persistence_diagram(persistence, axes=axes[1], legend=True)
axes[0].set_title("Barcode"); axes[1].set_title("Persistence diagram")
plt.tight_layout(); plt.show()

### Exercise 1 — When does a class die?
Change `fill_time` from 4 to 5 below and rerun. Which interval changes? Does $H_0$ change?
Then try `fill_time=2`: why does the positive-length $H_1$ interval disappear?

**Connection to Betti numbers:** count intervals satisfying $b\leq t<d$ to recover the number of classes alive at time $t$. The barcode also records which classes persist across times; Betti numbers alone do not determine it.

In [ ]:
# TODO: try 5.0, then 2.0, and explain the results.
fill_time = 4.0
changed = make_filtration(fill_time)
changed.persistence(homology_coeff_field=2)
print("H0:", changed.persistence_intervals_in_dimension(0))
print("H1:", changed.persistence_intervals_in_dimension(1))

for t in range(5):
    counts = []
    for k in [0, 1]:
        bars = st.persistence_intervals_in_dimension(k)
        counts.append(int(np.sum((bars[:, 0] <= t) & (t < bars[:, 1]))))
    print(f"t={t}: beta_0={counts[0]}, beta_1={counts[1]}")

**Your interpretation:**

_Write one or two sentences here._

## 2. From point clouds to Vietoris–Rips filtrations
In the lectures, an edge appears when $\|x_i-x_j\|\leq 2r$. **GUDHI uses the edge-length parameter $\alpha=2r$.** A simplex enters at its diameter (the longest edge). Divide GUDHI's birth and death coordinates by 2 if you want the lecture's radius convention; all following plots use $\alpha$.

We build the **2-skeleton**, including triangles, to compute $H_0$ and $H_1$. Edges alone cannot fill loops. This is a Rips construction, not the Čech nerve of balls: pairwise intersections do not guarantee a common intersection.

The small clouds below allow us to include all edges. Thus a surviving $H_1$ bar will not be an artefact of an early cutoff.

In [ ]:
theta = np.linspace(0, 2*np.pi, 24, endpoint=False)
circle = np.c_[np.cos(theta), np.sin(theta)]
two_circles = np.vstack([circle + [-1.6, 0], circle + [1.6, 0]])
rng = np.random.default_rng(SEED)
noisy_circle = circle + rng.normal(scale=.04, size=circle.shape)

clouds = {"One circle": circle, "Noisy circle": noisy_circle,
          "Two circles": two_circles}

def rips_persistence(points):
    # Include every edge; expand only up to triangles.
    cutoff = float(pdist(points).max()) + 1e-10
    tree = gd.RipsComplex(points=points, max_edge_length=cutoff).create_simplex_tree(
        max_dimension=2)
    diagram = tree.persistence(homology_coeff_field=2)
    return tree, diagram

rips_results = {name: rips_persistence(X) for name, X in clouds.items()}
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for col, (name, X) in enumerate(clouds.items()):
    axes[0, col].scatter(*X.T, s=16)
    axes[0, col].set(title=name, aspect="equal", xlim=(-3, 3), ylim=(-1.4, 1.4))
    tree, diagram = rips_results[name]
    h1 = [(k, interval) for k, interval in diagram if k == 1]
    gd.plot_persistence_barcode(h1, axes=axes[1, col])
    axes[1, col].set(title="$H_1$ barcode", xlabel=r"Edge length $\alpha$", xlim=(0, 2.2))
plt.tight_layout(); plt.show()

In [ ]:
# Draw a few snapshots of ONE Rips filtration, using its stored simplex times.
ring_tree = rips_results["One circle"][0]
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, alpha in zip(axes, [.2, .6, 1.8]):
    for simplex, time in ring_tree.get_filtration():
        if time > alpha:
            continue
        xy = circle[simplex]
        if len(simplex) == 3:
            ax.fill(*xy.T, color="teal", alpha=.025, zorder=0)
        elif len(simplex) == 2:
            ax.plot(*xy.T, color=".5", lw=.5, alpha=.4)
    ax.scatter(*circle.T, s=16, color="black")
    ax.set(title=fr"$\alpha={alpha}$ ($r={alpha/2}$)", aspect="equal")
    ax.axis("off")
plt.tight_layout(); plt.show()

### Exercise 2 — What does the long bar represent?
Identify the long $H_1$ intervals for one and two circles. Why do they eventually die even though we sampled circles?

Use the next cell to compare `max_dimension=1` and `max_dimension=2`. **Keep `persistence_dim_max=True` in the graph case**: otherwise GUDHI omits persistence in the complex's maximal dimension by default.

In [ ]:
# TODO: change 1 to 2. Why does the number of infinite H1 bars change?
max_dimension = 1
trial = gd.RipsComplex(points=circle, max_edge_length=2.1).create_simplex_tree(
    max_dimension=max_dimension)
trial.persistence(homology_coeff_field=2, persistence_dim_max=True)
h1_trial = trial.persistence_intervals_in_dimension(1)
print("Number of H1 bars:", len(h1_trial))
print("Infinite H1 bars:", np.isinf(h1_trial[:, 1]).sum())

**Your interpretation:**

_Write one or two sentences here._

## 3. Distances between barcodes
We compute distances using the equivalent **persistence diagrams**, one homological degree at a time. A matching can send a point to the diagonal, allowing diagrams with different numbers of bars to be compared.

- **Bottleneck:** minimize the largest matching cost, using the $\ell^\infty$ norm.
- **Wasserstein:** minimize the sum of costs to a power, then take the corresponding root. In GUDHI, `order` is the aggregation exponent and `internal_p` specifies the ground norm.

To match our definition of $W_p$, both parameters must agree, hence use `order=p, internal_p=p`. Other conventions often fix `internal_p=np.inf`; always state which one you use.

### Exercise 3 — A short extra bar
Let $A=\{(0,2)\}$ and $B=\{(0,2),(3,3.4)\}$. Predict the cost of matching the extra bar to the diagonal. Its closest diagonal point is $(3.2,3.2)$.

Compute $d_B$, $W_1$ with $\ell^1$ ground norm, and $W_2$ with Euclidean ground norm. Why are the answers different?

In [ ]:
A = np.array([[0., 2.]])
B = np.array([[0., 2.], [3., 3.4]])
print("Bottleneck:", gd.bottleneck_distance(A, B, e=0))
print("W1, l1 ground:", wasserstein_distance(A, B, order=1, internal_p=1))
w2, matching = wasserstein_distance(A, B, order=2, internal_p=2, matching=True)
print("W2, l2 ground:", w2)
print("Matching (A index, B index); -1 denotes the diagonal:\n", matching)

fig, ax = plt.subplots(figsize=(5, 4))
ax.plot([0, 4], [0, 4], "--", color=".6", label="Diagonal")
ax.scatter(*B.T, s=100, facecolors="none", edgecolors="teal", label="B")
ax.scatter(*A.T, marker="x", s=50, color="purple", label="A")
for i, j in matching:
    x = A[i] if i >= 0 else np.repeat(B[j].mean(), 2)
    y = B[j] if j >= 0 else np.repeat(A[i].mean(), 2)
    ax.plot([x[0], y[0]], [x[1], y[1]], color="orange", lw=2)
ax.set(xlabel="Birth", ylabel="Death", title="An optimal W2 matching", aspect="equal")
ax.legend(); plt.tight_layout(); plt.show()

**Your interpretation:**

_Write one or two sentences here._

### Compare the computed circle barcodes
Compare the clean circle with its noisy version and with two circles. We use only $H_1$: all intervals are finite because the filtrations include every edge and triangle. We do not silently discard infinite bars. An essential $H_0$ bar could be included if the two diagrams have compatible essential parts.

In [ ]:
D1 = {name: tree.persistence_intervals_in_dimension(1)
      for name, (tree, _) in rips_results.items()}
assert all(np.isfinite(d).all() for d in D1.values())
for name in ["Noisy circle", "Two circles"]:
    X, Y = D1["One circle"], D1[name]
    print(f"One circle vs {name}:")
    print("  bottleneck =", round(gd.bottleneck_distance(X, Y, e=0), 4))
    print("  W2 (l2)   =", round(wasserstein_distance(X, Y, order=2, internal_p=2), 4))
# TODO: increase the noise scale in Section 2, rerun that section and this cell.
# Does a small diagram distance imply that the original point clouds are identical?

**Your interpretation:**

_Write one or two sentences here._